In [1]:
# install if needed: uncomment the next line
# %pip install yfinance -q

import yfinance as yf
import matplotlib.pyplot as plt

In [2]:
ticker = input("Ticker (default AAPL): ") or "AAPL"
period = "1y"

df = yf.download(ticker, period=period, progress=False)
if df.empty:
    print("No data for", ticker)
else:
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(df.index, df["Close"], color="tab:blue", label="Close")
    ax1.set_ylabel("Price", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.bar(df.index, df["Volume"], alpha=0.2, color="tab:gray", label="Volume")
    ax2.set_ylabel("Volume", color="tab:gray")
    ax2.tick_params(axis="y", labelcolor="tab:gray")

    ax1.set_title(f"{ticker.upper()} Price and Volume ({period})")
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    plt.show()


1 Failed download:
- AAPL: No data found for this date range, symbol may be delisted
No data for AAPL


In [16]:
dat = yf.Ticker("BA")
print(dat.info)
print(dat.calendar)
# dat.analyst_price_targets
# dat.quarterly_income_stmt
# dat.history(period="1mo")
# dat.option_chain(dat.options[0]).calls


{'regularMarketPrice': None, 'preMarketPrice': None, 'logo_url': ''}
None


In [5]:
.get_dividends()

- AAPL: No data found for this date range, symbol may be delisted


[]

In [17]:
import os
import requests
import pandas as pd
from datetime import datetime

# Fetch Apple price history from Alpha Vantage and plot it.

API_KEY = "QYAFTAN0CIWPRIVY"
symbol = "AAPL"
url = "https://www.alphavantage.co/query"
params = {
    "function": "TIME_SERIES_DAILY_ADJUSTED",
    "symbol": symbol,
    "outputsize": "full",
    "apikey": API_KEY,
}

resp = requests.get(url, params=params, timeout=15)
data = resp.json()

if "Error Message" in data:
    raise ValueError("Alpha Vantage error: " + data["Error Message"])
if not any("Time Series" in k for k in data):
    raise ValueError("Unexpected API response: " + str(data))

if "Note" in data:
    print("Notice from API:", data["Note"])

ts_key = next(k for k in data.keys() if "Time Series" in k)
ts = data[ts_key]

av_df = pd.DataFrame.from_dict(ts, orient="index").rename(columns={
    "1. open": "Open",
    "2. high": "High",
    "3. low": "Low",
    "4. close": "Close",
    "5. adjusted close": "Adj Close",
    "6. volume": "Volume",
    "7. dividend amount": "Dividend Amount",
    "8. split coefficient": "Split Coef",
})
av_df.index = pd.to_datetime(av_df.index)
av_df = av_df.apply(pd.to_numeric, errors="coerce").sort_index()

# limit to last year if `period` exists and is like '1y'; otherwise default to last 365 days
days = 365
if globals().get("period") and isinstance(globals()["period"], str) and globals()["period"].endswith("y"):
    try:
        days = int(globals()["period"][:-1]) * 365
    except Exception:
        days = 365

df = av_df.last(f"{days}D")  # overwrite notebook df with Alpha Vantage history for convenience
print(df.tail())

# plot price and volume (uses matplotlib imported earlier in the notebook)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(df.index, df["Close"], color="tab:blue", label="Close")
ax1.set_ylabel("Price", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.bar(df.index, df["Volume"], alpha=0.2, color="tab:gray", label="Volume")
ax2.set_ylabel("Volume", color="tab:gray")
ax2.tick_params(axis="y", labelcolor="tab:gray")

ax1.set_title(f"{symbol} Price and Volume (last {days} days)")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.show()

ValueError: Unexpected API response: {'Information': 'Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium endpoints'}